# Team Assignment 8 - ML Deployment 

## Part 1 

Watch pre-class videos:
 - Overview of ML Deployment
 - Docker Containers [Code Demo Walkthrough]

## Part 2 

Put together a presentation tutorial (<7 mins) of your assigned tool/framework/topic. The tutorial should go end-to-end on deployment of a toy ML model. You will present these in class. Remember, you are teaching the class your topic so your presentation should be both educational and engaging. 

Bonus points are available if you optimize your model for deployment (e.g., through quantization or using ONNX for model inference).

Submission

Your submission will be your in-class tutorial presentation. 

Rubric

Presentation (25 points)

- Presentation is clear
- Presentation is engaging
- Presentation is <7 minutes
- Tutorial is clearly planned and well-thought out
- Transitions are handled gracefully
- Approach is easy to follow and understand

## Serverless Deployment using Azure Functions


⛔️ Followed instructions for VS Code deployment here:  https://learn.microsoft.com/en-us/azure/azure-functions/functions-create-function-app-portal?pivots=programming-language-python ... 
  - this did not work out, despite repeated attempts. Failures to create resources due to region (no functions can be created in US east with a paid account???), inexplicable failure to find and load triggers, ....

✅ Trying again, using exclusively (hopefully) the pay-as-you-go account `8cb3b59c-c4b0-4de8-8102-804c7463e8ca`. 

Pivoted to CLI-based creation given the challenges and inscrutability of the VS code method. See
https://learn.microsoft.com/en-us/cli/azure/install-azure-cli-macos. 

1. installed azure cli, this makes the `az` command available for interacting with azure resources

    ```
    brew update && brew install azure-cli
    ```
   
3. install core tools, this makes the `func` command available to interact with azure core tools

    ```
    brew tap azure/functions
    brew install azure-functions-core-tools@4
    # if upgrading on a machine that has 2.x or 3.x installed:
    brew link --overwrite azure-functions-core-tools@4
    ```
3. created a new venv `python3 -m venv .venv`, python 3.12 virtual environment
3. created a function project ... `func init --python` ...

    ```
    Found Python version 3.11.10 (python3.11).
    The new Python programming model is generally available. Learn more at https://aka.ms/pythonprogrammingmodel
    Writing requirements.txt
    Writing .funcignore
    Writing function_app.py
    .gitignore already exists. Skipped!
    Writing host.json
    Writing local.settings.json
    /Users/jason/Local/school/aipi510-fall24/.vscode/extensions.json already exists. Skipped!
    (.venv) jason@JasonsOficeMini aipi510-fall24 % python --version
    Python 3.12.4
    ```
    - ❗️ note the core tools CLI finds python 3.11, which is annoying as the venv i just created is 3.12 (what is it pointing at? this might cause us problems down the road. proceed for now and see what happens.
5. created a function within the project:

    ```
    (.venv) jason@JasonsOficeMini aipi510-fall24 % func new --name TrackstarsHttp --template "HTTP trigger" --authlevel "anonymous"  
    Appending to /Users/jason/Local/school/aipi510-fall24/function_app.py
    The function "TrackstarsHttp" was created successfully from the "HTTP trigger" template.
    ```
7. attempted to run as instructed ... `func start` and received a cascade of errors... it seems the app uses `Anonymous` as the access level but the actual syntax is `ANONYMOUS` per https://stackoverflow.com/questions/78668008/exception-attributeerror-anonymous-when-starting-a-basic-function-app. This resolves the issue:

    ```
    (.venv) jason@JasonsOficeMini aipi510-fall24 % func start
    Found Python version 3.11.10 (python3.11).
    
    Azure Functions Core Tools
    Core Tools Version:       4.0.6280 Commit hash: N/A +421f0144b42047aa289ce691dc6db4fc8b6143e6 (64-bit)
    Function Runtime Version: 4.834.3.22875
    
    [2024-11-09T14:28:40.758Z] Worker process started and initialized.
    
    Functions:
    
            TrackstarsHttp:  http://localhost:7071/api/TrackstarsHttp
    
    For detailed output, run func with --verbose flag.
    [2024-11-09T14:28:45.710Z] Host lock lease acquired by instance ID '0000000000000000000000008A9E5EBC'.
    [2024-11-09T14:29:02.240Z] Executing 'Functions.TrackstarsHttp' (Reason='This function was programmatically called via the host APIs.', Id=ecc258f4-b35f-43fd-9884-aa5ee4d59ecc)
    [2024-11-09T14:29:02.299Z] Python HTTP trigger function processed a request.
    [2024-11-09T14:29:02.350Z] Executed 'Functions.TrackstarsHttp' (Succeeded, Id=ecc258f4-b35f-43fd-9884-aa5ee4d59ecc, Duration=125ms)
    ```
9. now proceeding with the azure deployment, start by logging in...

    ```
    (.venv) jason@JasonsOficeMini aipi510-fall24 % az login
    A web browser has been opened at https://login.microsoftonline.com/organizations/oauth2/v2.0/authorize. Please continue the login in the web browser. If no web browser is available or if the web browser fails to open, use device code flow with `az login --use-device-code`.
    
    Retrieving tenants and subscriptions for the selection...
    
    [Tenant and subscription selection]
    
    No     Subscription name     Subscription ID                       Tenant
    -----  --------------------  ------------------------------------  -----------------
    [1] *  Pay-As-You-Go         8cb3b59c-c4b0-4de8-8102-804c7463e8ca  Default Directory
    [2]    Primary Subscription  52e0c49d-b9aa-483b-873c-a8d361238ed9  Default Directory
    
    The default is marked with an *; the default tenant is 'Default Directory' and subscription is 'Pay-As-You-Go' (8cb3b59c-c4b0-4de8-8102-804c7463e8ca).
    
    Select a subscription and tenant (Type a number or Enter for no changes): 
    
    Tenant: Default Directory
    Subscription: Pay-As-You-Go (8cb3b59c-c4b0-4de8-8102-804c7463e8ca)
    
    [Announcements]
    With the new Azure CLI login experience, you can select the subscription you want to use more easily. Learn more about it and its configuration at https://go.microsoft.com/fwlink/?linkid=2271236
    
    If you encounter any problem, please open an issue at https://aka.ms/azclibug
    
    [Warning] The login output has been updated. Please be aware that it no longer displays the full list of available subscriptions by default.
    ```
10. created a resource group:

    ```
    (.venv) jason@JasonsOficeMini aipi510-fall24 % az group create --name TrackstarsFunctionApp-rg --location "Central US"
    {
      "id": "/subscriptions/8cb3b59c-c4b0-4de8-8102-804c7463e8ca/resourceGroups/TrackstarsFunctionApp-rg",
      "location": "centralus",
      "managedBy": null,
      "name": "TrackstarsFunctionApp-rg",
      "properties": {
        "provisioningState": "Succeeded"
      },
      "tags": null,
      "type": "Microsoft.Resources/resourceGroups"
    }
    ```
11. created a storage account:

    ```
    (.venv) jason@JasonsOficeMini aipi510-fall24 % az storage account create --name trackstarsstorage --location "Central US" --resource-group TrackstarsFunctionApp-rg --sku Standard_LRS 
    {
      "accessTier": "Hot",
      "accountMigrationInProgress": null,
      "allowBlobPublicAccess": false,
      "allowCrossTenantReplication": false,
      "allowSharedKeyAccess": null,
      "allowedCopyScope": null,
      "azureFilesIdentityBasedAuthentication": null,
      "blobRestoreStatus": null,
      "creationTime": "2024-11-09T14:40:58.468734+00:00",
      "customDomain": null,
      "defaultToOAuthAuthentication": null,
      "dnsEndpointType": null,
      "enableExtendedGroups": null,
      "enableHttpsTrafficOnly": true,
      "enableNfsV3": null,
      "encryption": {
        "encryptionIdentity": null,
        "keySource": "Microsoft.Storage",
        "keyVaultProperties": null,
        "requireInfrastructureEncryption": null,
        "services": {
          "blob": {
            "enabled": true,
            "keyType": "Account",
            "lastEnabledTime": "2024-11-09T14:40:58.874981+00:00"
          },
          "file": {
            "enabled": true,
            "keyType": "Account",
            "lastEnabledTime": "2024-11-09T14:40:58.874981+00:00"
          },
          "queue": null,
          "table": null
        }
      },
      "extendedLocation": null,
      "failoverInProgress": null,
      "geoReplicationStats": null,
      "id": "/subscriptions/8cb3b59c-c4b0-4de8-8102-804c7463e8ca/resourceGroups/TrackstarsFunctionApp-rg/providers/Microsoft.Storage/storageAccounts/trackstarsstorage",
      "identity": null,
      "immutableStorageWithVersioning": null,
      "isHnsEnabled": null,
      "isLocalUserEnabled": null,
      "isSftpEnabled": null,
      "isSkuConversionBlocked": null,
      "keyCreationTime": {
        "key1": "2024-11-09T14:40:58.624983+00:00",
        "key2": "2024-11-09T14:40:58.624983+00:00"
      },
      "keyPolicy": null,
      "kind": "StorageV2",
      "largeFileSharesState": null,
      "lastGeoFailoverTime": null,
      "location": "centralus",
      "minimumTlsVersion": "TLS1_0",
      "name": "trackstarsstorage",
      "networkRuleSet": {
        "bypass": "AzureServices",
        "defaultAction": "Allow",
        "ipRules": [],
        "ipv6Rules": [],
        "resourceAccessRules": null,
        "virtualNetworkRules": []
      },
      "primaryEndpoints": {
        "blob": "https://trackstarsstorage.blob.core.windows.net/",
        "dfs": "https://trackstarsstorage.dfs.core.windows.net/",
        "file": "https://trackstarsstorage.file.core.windows.net/",
        "internetEndpoints": null,
        "microsoftEndpoints": null,
        "queue": "https://trackstarsstorage.queue.core.windows.net/",
        "table": "https://trackstarsstorage.table.core.windows.net/",
        "web": "https://trackstarsstorage.z19.web.core.windows.net/"
      },
      "primaryLocation": "centralus",
      "privateEndpointConnections": [],
      "provisioningState": "Succeeded",
      "publicNetworkAccess": null,
      "resourceGroup": "TrackstarsFunctionApp-rg",
      "routingPreference": null,
      "sasPolicy": null,
      "secondaryEndpoints": null,
      "secondaryLocation": null,
      "sku": {
        "name": "Standard_LRS",
        "tier": "Standard"
      },
      "statusOfPrimary": "available",
      "statusOfSecondary": null,
      "storageAccountSkuConversionStatus": null,
      "tags": {},
      "type": "Microsoft.Storage/storageAccounts"
    }
    ```

10. created function app

    ```
    (.venv) jason@JasonsOficeMini aipi510-fall24 % az functionapp create --resource-group TrackstarsFunctionApp-rg --consumption-plan-location centralus --runtime python --runtime-version 3.11 --functions-version 4 --name TrackstarsFunctionApp --os-type linux --storage-account trackstarsstorage
    
    /opt/homebrew/Cellar/azure-cli/2.65.0_2/libexec/lib/python3.11/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
      "cipher": algorithms.TripleDES,
    /opt/homebrew/Cellar/azure-cli/2.65.0_2/libexec/lib/python3.11/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
      "class": algorithms.TripleDES,
    Your Linux function app 'TrackstarsFunctionApp', that uses a consumption plan has been successfully created but is not active until content is published using Azure Portal or the Functions Core Tools.
    Application Insights "TrackstarsFunctionApp" was created for this Function App. You can visit https://portal.azure.com/#resource/subscriptions/8cb3b59c-c4b0-4de8-8102-804c7463e8ca/resourceGroups/TrackstarsFunctionApp-rg/providers/microsoft.insights/components/TrackstarsFunctionApp/overview to view your Application Insights component
    App settings have been redacted. Use `az webapp/logicapp/functionapp config appsettings list` to view.
    {
      "availabilityState": "Normal",
      "clientAffinityEnabled": false,
      "clientCertEnabled": false,
      "clientCertExclusionPaths": null,
      "clientCertMode": "Required",
      "cloningInfo": null,
      "containerSize": 0,
      "customDomainVerificationId": "A723E10661FB802BADB852CADCB0FB33F8D20C6042F2CE9C6F7BAB419E9DB4F9",
      "dailyMemoryTimeQuota": 0,
      "daprConfig": null,
      "defaultHostName": "trackstarsfunctionapp.azurewebsites.net",
      "enabled": true,
      "enabledHostNames": [
        "trackstarsfunctionapp.azurewebsites.net",
        "trackstarsfunctionapp.scm.azurewebsites.net"
      ],
      "extendedLocation": null,
      "hostNameSslStates": [
        {
          "certificateResourceId": null,
          "hostType": "Standard",
          "ipBasedSslResult": null,
          "ipBasedSslState": "NotConfigured",
          "name": "trackstarsfunctionapp.azurewebsites.net",
          "sslState": "Disabled",
          "thumbprint": null,
          "toUpdate": null,
          "toUpdateIpBasedSsl": null,
          "virtualIPv6": null,
          "virtualIp": null
        },
        {
          "certificateResourceId": null,
          "hostType": "Repository",
          "ipBasedSslResult": null,
          "ipBasedSslState": "NotConfigured",
          "name": "trackstarsfunctionapp.scm.azurewebsites.net",
          "sslState": "Disabled",
          "thumbprint": null,
          "toUpdate": null,
          "toUpdateIpBasedSsl": null,
          "virtualIPv6": null,
          "virtualIp": null
        }
      ],
      "hostNames": [
        "trackstarsfunctionapp.azurewebsites.net"
      ],
      "hostNamesDisabled": false,
      "hostingEnvironmentProfile": null,
      "httpsOnly": false,
      "hyperV": false,
      "id": "/subscriptions/8cb3b59c-c4b0-4de8-8102-804c7463e8ca/resourceGroups/TrackstarsFunctionApp-rg/providers/Microsoft.Web/sites/TrackstarsFunctionApp",
      "identity": null,
      "inProgressOperationId": null,
      "isDefaultContainer": null,
      "isXenon": false,
      "keyVaultReferenceIdentity": "SystemAssigned",
      "kind": "functionapp,linux",
      "lastModifiedTimeUtc": "2024-11-09T14:44:12.333333",
      "location": "centralus",
      "managedEnvironmentId": null,
      "maxNumberOfWorkers": null,
      "name": "TrackstarsFunctionApp",
      "outboundIpAddresses": "40.122.169.200,104.208.27.4,40.122.42.64,23.99.130.19,13.89.172.4",
      "possibleOutboundIpAddresses": "40.122.169.200,104.208.27.4,40.122.42.64,23.99.130.19,13.67.176.18,104.208.28.216,40.122.168.112,40.122.169.80,23.99.159.248,20.118.17.81,20.118.19.166,20.118.19.182,20.118.20.77,20.118.20.179,20.118.20.230,20.118.21.220,20.118.22.146,20.118.22.148,20.118.22.161,20.118.22.177,20.118.22.227,13.89.172.4",
      "publicNetworkAccess": null,
      "redundancyMode": "None",
      "repositorySiteName": "TrackstarsFunctionApp",
      "reserved": true,
      "resourceConfig": null,
      "resourceGroup": "TrackstarsFunctionApp-rg",
      "scmSiteAlsoStopped": false,
      "serverFarmId": "/subscriptions/8cb3b59c-c4b0-4de8-8102-804c7463e8ca/resourceGroups/TrackstarsFunctionApp-rg/providers/Microsoft.Web/serverfarms/CentralUSLinuxDynamicPlan",
      "siteConfig": {
        "acrUseManagedIdentityCreds": false,
        "acrUserManagedIdentityId": null,
        "alwaysOn": false,
        "antivirusScanEnabled": null,
        "apiDefinition": null,
        "apiManagementConfig": null,
        "appCommandLine": null,
        "appSettings": null,
        "autoHealEnabled": null,
        "autoHealRules": null,
        "autoSwapSlotName": null,
        "azureMonitorLogCategories": null,
        "azureStorageAccounts": null,
        "clusteringEnabled": false,
        "connectionStrings": null,
        "cors": null,
        "customAppPoolIdentityAdminState": null,
        "customAppPoolIdentityTenantState": null,
        "defaultDocuments": null,
        "detailedErrorLoggingEnabled": null,
        "documentRoot": null,
        "elasticWebAppScaleLimit": null,
        "experiments": null,
        "fileChangeAuditEnabled": null,
        "ftpsState": null,
        "functionAppScaleLimit": 0,
        "functionsRuntimeScaleMonitoringEnabled": null,
        "handlerMappings": null,
        "healthCheckPath": null,
        "http20Enabled": false,
        "http20ProxyFlag": null,
        "httpLoggingEnabled": null,
        "ipSecurityRestrictions": [
          {
            "action": "Allow",
            "description": "Allow all access",
            "headers": null,
            "ipAddress": "Any",
            "name": "Allow all",
            "priority": 2147483647,
            "subnetMask": null,
            "subnetTrafficTag": null,
            "tag": null,
            "vnetSubnetResourceId": null,
            "vnetTrafficTag": null
          }
        ],
        "ipSecurityRestrictionsDefaultAction": null,
        "javaContainer": null,
        "javaContainerVersion": null,
        "javaVersion": null,
        "keyVaultReferenceIdentity": null,
        "limits": null,
        "linuxFxVersion": "",
        "loadBalancing": null,
        "localMySqlEnabled": null,
        "logsDirectorySizeLimit": null,
        "machineKey": null,
        "managedPipelineMode": null,
        "managedServiceIdentityId": null,
        "metadata": null,
        "minTlsCipherSuite": null,
        "minTlsVersion": null,
        "minimumElasticInstanceCount": 0,
        "netFrameworkVersion": null,
        "nodeVersion": null,
        "numberOfWorkers": 1,
        "phpVersion": null,
        "powerShellVersion": null,
        "preWarmedInstanceCount": null,
        "publicNetworkAccess": null,
        "publishingPassword": null,
        "publishingUsername": null,
        "push": null,
        "pythonVersion": null,
        "remoteDebuggingEnabled": null,
        "remoteDebuggingVersion": null,
        "requestTracingEnabled": null,
        "requestTracingExpirationTime": null,
        "routingRules": null,
        "runtimeADUser": null,
        "runtimeADUserPassword": null,
        "scmIpSecurityRestrictions": [
          {
            "action": "Allow",
            "description": "Allow all access",
            "headers": null,
            "ipAddress": "Any",
            "name": "Allow all",
            "priority": 2147483647,
            "subnetMask": null,
            "subnetTrafficTag": null,
            "tag": null,
            "vnetSubnetResourceId": null,
            "vnetTrafficTag": null
          }
        ],
        "scmIpSecurityRestrictionsDefaultAction": null,
        "scmIpSecurityRestrictionsUseMain": null,
        "scmMinTlsCipherSuite": null,
        "scmMinTlsVersion": null,
        "scmSupportedTlsCipherSuites": null,
        "scmType": null,
        "sitePort": null,
        "sitePrivateLinkHostEnabled": null,
        "storageType": null,
        "supportedTlsCipherSuites": null,
        "tracingOptions": null,
        "use32BitWorkerProcess": null,
        "virtualApplications": null,
        "vnetName": null,
        "vnetPrivatePortsCount": null,
        "vnetRouteAllEnabled": null,
        "webSocketsEnabled": null,
        "websiteTimeZone": null,
        "winAuthAdminState": null,
        "winAuthTenantState": null,
        "windowsConfiguredStacks": null,
        "windowsFxVersion": null,
        "xManagedServiceIdentityId": null
      },
      "slotSwapStatus": null,
      "state": "Running",
      "storageAccountRequired": false,
      "suspendedTill": null,
      "tags": null,
      "targetSwapSlot": null,
      "trafficManagerHostNames": null,
      "type": "Microsoft.Web/sites",
      "usageState": "Normal",
      "virtualNetworkSubnetId": null,
      "vnetContentShareEnabled": false,
      "vnetImagePullEnabled": false,
      "vnetRouteAllEnabled": false,
      "workloadProfileName": null
    }
    ```
